# 20. Valid Parentheses
**Difficulty:** 🟢 Easy · **Topic:** String · **LeetCode:** https://leetcode.com/problems/valid-parentheses/

## 💡 Concepts

**Core concept(s):** A **stack** — the most-recent opener must be the first to close.

**Why it applies here:** Brackets nest like boxes inside boxes. The last box you open is the first you must close. A stack remembers the openers in the exact order needed and hands back the most recent one when a closer arrives.

**Key intuition:** Push every opener; when a closer comes, it must match the opener on top of the stack.

---

### 📚 What is a Stack?
A **stack** is a "last in, first out" pile — like a stack of plates. You **push** onto the top and **pop** from the top.
- **Complexity:** push / pop / peek are **O(1)**.
- **In Python:** a plain `list`; `append(x)` to push, `pop()` to remove the top, `list[-1]` to peek.

---

**Prerequisite knowledge:**
- Using a `list` as a stack (`append`, `pop`, `[-1]`).

## 📝 Problem

Given a string of `()[]{}`, return `True` if every bracket is closed by the correct type in the correct order.

**Example**
```
"()[]{}" -> True
"(]"     -> False
"([)]"   -> False
"{[]}"   -> True
```

### Approach 1 — Repeatedly Remove Pairs (worst)

**Idea:** Keep deleting adjacent matching pairs (`()`, `[]`, `{}`) until nothing changes. Valid iff the string becomes empty.

**Time complexity:** `O(n^2)` — each cleanup pass is O(n) and we may need O(n) passes.

**Space complexity:** `O(n)`.

In [ ]:
def valid_paren_brute(s: str) -> bool:
    prev = None
    # Keep deleting adjacent matching pairs until the string stops changing.
    while prev != s:                       # loop again only if last pass removed something
        prev = s                           # remember the string before this cleanup pass
        s = s.replace("()", "").replace("[]", "").replace("{}", "")  # strip simple pairs
    return s == ""                         # valid only if everything cancelled out

### Approach 2 — Stack (optimal)

**Idea:** Push each opener. On a closer, the top of the stack must be its matching opener — pop it. If it isn't (or the stack is empty), it's invalid. At the end the stack must be empty.

**Time complexity:** `O(n)`.

**Space complexity:** `O(n)` for the stack.

In [ ]:
def valid_paren_stack(s: str) -> bool:
    match = {")": "(", "]": "[", "}": "{"}  # each closer mapped to the opener it needs
    stack = []                             # holds the openers we've seen but not yet closed
    for c in s:                            # scan the string left to right
        if c in match:                     # c is a CLOSER
            # It's valid only if the most recent opener (top of stack) matches.
            if not stack or stack[-1] != match[c]:
                return False               # nothing to close, or wrong type -> invalid
            stack.pop()                    # matched -> remove that opener
        else:                              # c is an OPENER
            stack.append(c)                # remember it until its closer arrives
    return not stack                       # valid only if no opener was left unclosed

In [ ]:
# Correctness check
tests = [("()[]{}",True), ("(]",False), ("([)]",False), ("{[]}",True), ("(",False), ("",True)]
for s, exp in tests:
    a, b = valid_paren_brute(s), valid_paren_stack(s)
    print(f"{s!r:>10} -> brute={a}, stack={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    half = n // 2
    s = "(" * half + ")" * half             # deeply nested & valid -> worst for both
    return (s,)

solutions = {
    "brute O(n^2)": valid_paren_brute,
    "stack O(n)  ": valid_paren_stack,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Stack for nesting / "most recent must resolve first":** whenever the latest thing opened must be the first closed, a stack is the natural tool.
- **Signal:** "balanced brackets", "matching pairs", "valid nesting", "undo the last".
- **Related problems:** Min Stack, Generate Parentheses, Evaluate Reverse Polish Notation, Daily Temperatures.
- **Common pitfalls:** (1) forgetting to check the stack is empty at the end; (2) popping an empty stack on a stray closer.